In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Butterfly Effect\n",
    "## Sensitivity to Initial Conditions\n",
    "\n",
    "Two plaintexts differing by one bit produce utterly different ciphertexts.\n",
    "The chaotic network amplifies tiny changes exponentially."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "from network import ChaoticOscillatoryNetwork\n",
    "from crypto import CAMCCipher\n",
    "from dynamics import divergence_curve"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "plt.rcParams.update({\n",
    "    'figure.facecolor': '#fafafa',\n",
    "    'axes.facecolor': '#fafafa',\n",
    "    'axes.edgecolor': '#222222',\n",
    "    'axes.labelcolor': '#222222',\n",
    "    'text.color': '#222222',\n",
    "    'font.size': 11,\n",
    "    'axes.grid': True,\n",
    "    'grid.alpha': 0.3,\n",
    "    'grid.color': '#999999'\n",
    "})"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "N = 128\n",
    "SCALE = 2.0\n",
    "\n",
    "np.random.seed(7)\n",
    "network = ChaoticOscillatoryNetwork(n_neurons=N, weight_scale=SCALE)\n",
    "sync = np.exp(1j * np.random.uniform(0, 2 * np.pi, N))\n",
    "cipher = CAMCCipher(network, sync)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "plain_a = \"Butterfly effect demonstration.\"\n",
    "plain_b = \"Butterfly effect demonsttation.\"\n",
    "\n",
    "enc_a = cipher.encrypt(plain_a)\n",
    "enc_b = cipher.encrypt(plain_b)\n",
    "\n",
    "diff_bits = sum(bin(x ^ y).count('1') for x, y in zip(enc_a, enc_b))\n",
    "total_bits = len(enc_a) * 8\n",
    "print(f\"Changed bits: {diff_bits}/{total_bits} = {100*diff_bits/total_bits:.1f}%\")\n",
    "print(f\"Cipher A hex: {enc_a.hex()}\")\n",
    "print(f\"Cipher B hex: {enc_b.hex()}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "state_a = cipher.sync_state.copy()\n",
    "state_b = state_a.copy()\n",
    "state_b[0] *= np.exp(1j * 1e-4)\n",
    "\n",
    "distances = divergence_curve(network, state_a, state_b, steps=150)\n",
    "\n",
    "fig, ax = plt.subplots(figsize=(8, 3))\n",
    "ax.plot(distances, color='#4a7c59')\n",
    "ax.set_yscale('log')\n",
    "ax.set_xlabel('Step')\n",
    "ax.set_ylabel('Euclidean distance (log scale)')\n",
    "ax.set_title('Divergence of two trajectories')\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "dec_a = cipher.decrypt(enc_a)\n",
    "dec_b = cipher.decrypt(enc_b)\n",
    "print(f\"Decrypted A: {dec_a}\")\n",
    "print(f\"Decrypted B: {dec_b}\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.10.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}